# RgDAAO–ESM-2 benchmark
Select **Runtime → Change runtime type → T4 GPU** (or another available NVIDIA GPU), then run all cells in a fresh runtime.
This notebook runs the repository implementation directly and compares the frozen ESM-2 baseline with supervised fine-tuning on the same fixed split.
The final cell downloads a results archive; retain it before the Colab runtime expires.
A fresh runtime is recommended for a new benchmark run so stale results cannot be mixed with a new experiment.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path
repo = Path('/content/RgDAAO-ESM2')
if repo.exists():
    shutil.rmtree(repo)
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/jjimenezgar/RgDAAO-ESM2.git', str(repo)], check=True)
os.chdir(repo)
print('Repository commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[ml]'], check=True)
import torch
assert torch.cuda.is_available(), 'GPU not detected. Select Runtime → Change runtime type → T4 GPU, then restart this notebook.'
print('GPU:', torch.cuda.get_device_name(0))
for script in ('download_data.py', 'build_dataset.py'):
    subprocess.run([sys.executable, 'scripts/' + script], check=True)
subprocess.run([sys.executable, '-m', 'rgdaao.prepare', '--input', 'data/processed/variants.csv', '--output', 'data/processed'], check=True)
print('Dataset preparation complete.')

In [ ]:
import shutil
from google.colab import files

def run_visible(command):
    print('$', ' '.join(map(str, command)), flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='')
    returncode = process.wait()
    if returncode:
        raise RuntimeError(f'Command failed with exit code {returncode}: {command}')

print('Running ESM-2 software smoke test...')
run_visible([sys.executable, 'scripts/smoke_test.py'])
print('Smoke test passed. Starting the real frozen vs fine-tuned benchmark...')
run_visible(['bash', 'scripts/run_benchmark.sh'])
shutil.copytree('data/processed', 'results/data_audit', dirs_exist_ok=True)
# Retain selected models and predictions; intermediate optimizer checkpoints are unnecessary for inference.
for folder in Path('results').glob('esm2_*/checkpoints'):
    shutil.rmtree(folder)
archive = shutil.make_archive('/content/RgDAAO-ESM2-results', 'zip', 'results')
print('Benchmark complete. Results archive:', archive)
files.download(archive)

Each experiment saves `config.json`, `environment.json`, `pip-freeze.txt`, `run.json`, `history.json`, the selected checkpoint, `metrics.json` and `predictions.csv`.
`results/comparison.csv` and `results/figures/activity_predictions.{png,pdf}` are created only from completed test evaluations.
Share the downloaded archive to inspect and document the real comparison. The smoke report is a software check, not scientific performance.